# Введение в MapReduce модель на Python


In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [ ]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [ ]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [ ]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [ ]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [ ]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [ ]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [ ]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [ ]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [ ]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [ ]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [ ]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, 2.905589809636405),
 (1, 2.905589809636405),
 (2, 2.905589809636405),
 (3, 2.905589809636405),
 (4, 2.905589809636405)]

## Inverted index

In [ ]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [ ]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [ ]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [ ]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('banana', 2), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [])]

## TeraSort

In [ ]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, 0.0059671639991895065),
   (None, 0.07724245496172),
   (None, 0.08440804135613444),
   (None, 0.13575647907181598),
   (None, 0.14404826813474803),
   (None, 0.21204275967955666),
   (None, 0.21869633101751806),
   (None, 0.25055756276216923),
   (None, 0.28642389538931257),
   (None, 0.3834487515438496),
   (None, 0.3913614390023946),
   (None, 0.4041378102237341),
   (None, 0.41854626274930695),
   (None, 0.4704310153549396),
   (None, 0.4776995227348928),
   (None, 0.48992216726013693)]),
 (1,
  [(None, 0.5005353544023048),
   (None, 0.5135664686748047),
   (None, 0.53391984417089),
   (None, 0.5587932025401512),
   (None, 0.5673804905854288),
   (None, 0.6926646597910275),
   (None, 0.7237444251339501),
   (None, 0.7557883138083207),
   (None, 0.785709769245918),
   (None, 0.7937098630029404),
   (None, 0.7942646850708935),
   (None, 0.9160468126494941),
   (None, 0.9618068292060864),
   (None, 0.9820764489731459)])]

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
from typing import Iterator, NamedTuple
import random

input_numbers = [random.randint(1, 100) for _ in range(20)]
print(f"Входные числа: {input_numbers}")

def RECORDREADER():
    for i, num in enumerate(input_numbers):
        yield (i, num) # пары индекс-число

def MAP(key: int, value: int):
    yield ('max', value)

def REDUCE(key: str, values: Iterator[int]):
    max_value = float('-inf')  # Начинаем с min ищем max
    for v in values:
        if v > max_value:
            max_value = v
    yield (key, max_value)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x),
                       groupbykey(flatten(map(lambda x: MAP(*x),
                                             RECORDREADER())))))

result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Результат MapReduce: {result}")
print(f"Проверка обычным max: {max(input_numbers)}")

Входные числа: [77, 8, 93, 53, 19, 8, 46, 45, 68, 70, 70, 6, 75, 78, 93, 88, 39, 42, 75, 13]
Результат MapReduce: [('max', 93)]
Проверка обычным max: 93


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [ ]:
from typing import Iterator, NamedTuple, Tuple
import random

input_numbers = [random.randint(1, 100) for _ in range(20)]
print(f"Входные числа: {input_numbers}")
print(f"Среднее арифметическое: {sum(input_numbers)/len(input_numbers)}")

def RECORDREADER():
    for i, num in enumerate(input_numbers):
        yield (i, num)

def MAP(key: int, value: int):
    yield ('avg', (value, 1))

def REDUCE(key: str, values: Iterator[Tuple[int, int]]):
    total_sum = 0
    total_count = 0

    for num, count in values:
        total_sum += num
        total_count += count

    if total_count > 0:
        average = total_sum / total_count
        yield (key, average)
    else:
        yield (key, 0)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x),
                       groupbykey(flatten(map(lambda x: MAP(*x),
                                             RECORDREADER())))))

result = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(f"Результат MapReduce: {result}")

Входные числа: [18, 74, 31, 47, 33, 87, 79, 22, 79, 68, 44, 98, 72, 85, 75, 80, 74, 80, 60, 58]
Среднее арифметическое: 63.2
Результат MapReduce: [('avg', 63.2)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [ ]:
from typing import Iterator, List, Tuple, Any
import random

def groupbykey_sort_based(iterable: Iterator[Tuple[Any, Any]]) -> Iterator[Tuple[Any, List[Any]]]:
    sorted_pairs = sorted(iterable, key=lambda x: x[0])

    if not sorted_pairs:
        return

    current_key = sorted_pairs[0][0]
    current_values = []

    for key, value in sorted_pairs:
        if key == current_key:
            current_values.append(value)
        else:
            yield (current_key, current_values)
            current_key = key
            current_values = [value]

    yield (current_key, current_values)

def MapReduce_with_sort(RECORDREADER, MAP, REDUCE):
    map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
    grouped = groupbykey_sort_based(map_output)
    return flatten(map(lambda x: REDUCE(*x), grouped))

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

# Тестовые данные: пары (ключ, значение)
test_pairs = [
    ('b', 2),
    ('a', 1),
    ('c', 3),
    ('b', 4),
    ('a', 5),
    ('c', 6),
    ('a', 7)
]

print("Исходные пары:", test_pairs)
print("\nРезультат groupByKey на основе сортировки:")

result = list(groupbykey_sort_based(test_pairs))
for key, values in result:
    print(f"  {key}: {values}")

documents = [
    "hello world hello",
    "world mapreduce hello",
    "mapreduce world"
]

def wordcount_MAP(doc_id: int, text: str):
    for word in text.split():
        yield (word, 1)

def wordcount_REDUCE(word: str, counts: Iterator[int]):
    yield (word, sum(counts))

def wordcount_RECORDREADER():
    for i, doc in enumerate(documents):
        yield (i, doc)

print("\n")
print("WordCount с сортировочным groupByKey")

result = list(MapReduce_with_sort(wordcount_RECORDREADER, wordcount_MAP, wordcount_REDUCE))
print(f"Результат: {sorted(result)}")


Исходные пары: [('b', 2), ('a', 1), ('c', 3), ('b', 4), ('a', 5), ('c', 6), ('a', 7)]

Результат groupByKey на основе сортировки:
  a: [1, 5, 7]
  b: [2, 4]
  c: [3, 6]


WordCount с сортировочным groupByKey
Результат: [('hello', 3), ('mapreduce', 2), ('world', 3)]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [ ]:
from typing import Iterator, Any
import random
import hashlib

input_data = [1, 2, 3, 2, 1, 4, 5, 3, 6, 7, 5, 8, 9, 8, 7]
maps = 3
reducers = 2

print(f"Исходные данные ({len(input_data)} элементов): {input_data}")
print(f"Уникальных элементов: {len(set(input_data))}")

def INPUTFORMAT():
    global maps

    def RECORDREADER(split_data):
        for i, value in enumerate(split_data):
            yield (i, value)  # (индекс, значение)

    split_size = len(input_data) // maps + (1 if len(input_data) % maps else 0)
    for i in range(0, len(input_data), split_size):
        yield RECORDREADER(input_data[i:i+split_size])

def MAP(key: int, value: Any):
    yield (value, None)

def COMBINER(element: Any, _: Iterator[None]):
    yield (element, None)

def PARTITIONER(key: Any):
    global reducers
    return hash(key) % reducers

def REDUCE(element: Any, _: Iterator[None]):
    yield (element, None)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
    global reducers
    partitions = [dict() for _ in range(reducers)]
    for map_partition in map_partitions:
        for (k2, v2) in map_partition:
            p = partitions[PARTITIONER(k2)]
            p[k2] = p.get(k2, []) + [v2]
    return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
    map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
    if COMBINER != None:
        map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
    reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER)
    reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

    network_pairs = sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])
    print(f"По сети передано {network_pairs} пар ключ-значение")
    return reduce_outputs

print("Без COMBINER")

result_no_combiner = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=None)

print("\nРезультаты по партициям:")
unique_elements = []
for partition_id, partition_output in result_no_combiner:
    elements = list(partition_output)
    print(f"Партиция {partition_id}: {[e for e, _ in elements]}")
    unique_elements.extend([e for e, _ in elements])

print(f"\nВсего уникальных элементов: {len(unique_elements)}")
print(f"Уникальные элементы: {sorted(unique_elements)}")

print("\n")
print("С COMBINER")

result_with_combiner = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=COMBINER)

print("\nРезультаты по партициям:")
unique_elements = []
for partition_id, partition_output in result_with_combiner:
    elements = list(partition_output)
    print(f"Партиция {partition_id}: {[e for e, _ in elements]}")
    unique_elements.extend([e for e, _ in elements])

print(f"\nВсего уникальных элементов: {len(unique_elements)}")
print(f"Уникальные элементы: {sorted(unique_elements)}")

Исходные данные (15 элементов): [1, 2, 3, 2, 1, 4, 5, 3, 6, 7, 5, 8, 9, 8, 7]
Уникальных элементов: 9
Без COMBINER
По сети передано 15 пар ключ-значение

Результаты по партициям:
Партиция 0: [2, 4, 6, 8]
Партиция 1: [1, 3, 5, 7, 9]

Всего уникальных элементов: 9
Уникальные элементы: [1, 2, 3, 4, 5, 6, 7, 8, 9]


С COMBINER
По сети передано 12 пар ключ-значение

Результаты по партициям:
Партиция 0: [2, 4, 6, 8]
Партиция 1: [1, 3, 5, 7, 9]

Всего уникальных элементов: 9
Уникальные элементы: [1, 2, 3, 4, 5, 6, 7, 8, 9]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [ ]:
from typing import Iterator, NamedTuple, Any
import random

class User(NamedTuple):
    id: int
    name: str
    age: int
    city: str

# тестовые данные
users = [
    User(id=1, name="Alice", age=25, city="Moscow"),
    User(id=2, name="Bob", age=30, city="SPb"),
    User(id=3, name="Charlie", age=35, city="Moscow"),
    User(id=4, name="Diana", age=28, city="Kazan"),
    User(id=5, name="Eve", age=32, city="Moscow"),
    User(id=6, name="Frank", age=40, city="SPb"),
    User(id=7, name="Grace", age=22, city="Moscow"),
]

print("Исходные данные:")
for user in users:
    print(f"  {user}")

def RECORDREADER():
    for user in users:
        yield (user.id, user)

def selection_MAP(key: int, user: User):
    if user.age >= 30:
        yield (user, user)

def selection_REDUCE(user: User, values: Iterator[User]):
    yield (user, user)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x),
                       groupbykey(flatten(map(lambda x: MAP(*x),
                                             RECORDREADER())))))

print("SELECT * FROM users WHERE age >= 30")

result = list(MapReduce(RECORDREADER, selection_MAP, selection_REDUCE))
print(f"\nРезультат ({len(result)} записей):")
for key, user in result:
    print(f"  {user}")

Исходные данные:
  User(id=1, name='Alice', age=25, city='Moscow')
  User(id=2, name='Bob', age=30, city='SPb')
  User(id=3, name='Charlie', age=35, city='Moscow')
  User(id=4, name='Diana', age=28, city='Kazan')
  User(id=5, name='Eve', age=32, city='Moscow')
  User(id=6, name='Frank', age=40, city='SPb')
  User(id=7, name='Grace', age=22, city='Moscow')
SELECT * FROM users WHERE age >= 30

Результат (4 записей):
  User(id=2, name='Bob', age=30, city='SPb')
  User(id=3, name='Charlie', age=35, city='Moscow')
  User(id=5, name='Eve', age=32, city='Moscow')
  User(id=6, name='Frank', age=40, city='SPb')


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [ ]:
from typing import Iterator, NamedTuple, Any, Tuple, List
from dataclasses import dataclass
import json

class User(NamedTuple):
    id: int
    name: str
    age: int
    city: str
    occupation: str

users = [
    User(id=1, name="Alice", age=25, city="Moscow", occupation="engineer"),
    User(id=2, name="Bob", age=30, city="SPb", occupation="doctor"),
    User(id=3, name="Charlie", age=35, city="Moscow", occupation="teacher"),
    User(id=4, name="Diana", age=25, city="Kazan", occupation="engineer"),
    User(id=5, name="Eve", age=30, city="Moscow", occupation="doctor"),
    User(id=6, name="Frank", age=40, city="SPb", occupation="teacher"),
    User(id=7, name="Grace", age=25, city="Moscow", occupation="engineer"),
]

print("Исходные данные:")
for user in users:
    print(f"  {user}")

def RECORDREADER():
    for user in users:
        yield (user.id, user)

def create_projection_MAP(attributes: List[str]):
    def MAP(key: int, user: User):
        projected_dict = {}
        for attr in attributes:
            if hasattr(user, attr):
                projected_dict[attr] = getattr(user, attr)

        projected_tuple = tuple(sorted(projected_dict.items()))

        yield (projected_tuple, projected_tuple)

    return MAP

def projection_REDUCE(projected_tuple: Tuple, values: Iterator[Tuple]):
    yield (projected_tuple, projected_tuple)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x),
                       groupbykey(flatten(map(lambda x: MAP(*x),
                                             RECORDREADER())))))

print("Проекция на возраст")

project_age_MAP = create_projection_MAP(['age'])

result = list(MapReduce(RECORDREADER, project_age_MAP, projection_REDUCE))

print("Уникальные значения возраста:")
projected_values = []
for proj_tuple, _ in result:
    proj_dict = dict(proj_tuple)
    print(f"  {proj_dict}")
    projected_values.append(proj_dict['age'])

print(f"\nВсего уникальных значений: {len(projected_values)}")
print(f"Значения: {sorted(projected_values)}")

print("\n")
print("Проекция на (город, профессия)")

project_city_occ_MAP = create_projection_MAP(['city', 'occupation'])

result = list(MapReduce(RECORDREADER, project_city_occ_MAP, projection_REDUCE))

print("Уникальные комбинации (город, профессия):")
for proj_tuple, _ in sorted(result):
    proj_dict = dict(proj_tuple)
    print(f"  {proj_dict}")

print(f"\nВсего уникальных комбинаций: {len(result)}")

Исходные данные:
  User(id=1, name='Alice', age=25, city='Moscow', occupation='engineer')
  User(id=2, name='Bob', age=30, city='SPb', occupation='doctor')
  User(id=3, name='Charlie', age=35, city='Moscow', occupation='teacher')
  User(id=4, name='Diana', age=25, city='Kazan', occupation='engineer')
  User(id=5, name='Eve', age=30, city='Moscow', occupation='doctor')
  User(id=6, name='Frank', age=40, city='SPb', occupation='teacher')
  User(id=7, name='Grace', age=25, city='Moscow', occupation='engineer')
Проекция на возраст
Уникальные значения возраста:
  {'age': 25}
  {'age': 30}
  {'age': 35}
  {'age': 40}

Всего уникальных значений: 4
Значения: [25, 30, 35, 40]


Проекция на (город, профессия)
Уникальные комбинации (город, профессия):
  {'city': 'Kazan', 'occupation': 'engineer'}
  {'city': 'Moscow', 'occupation': 'doctor'}
  {'city': 'Moscow', 'occupation': 'engineer'}
  {'city': 'Moscow', 'occupation': 'teacher'}
  {'city': 'SPb', 'occupation': 'doctor'}
  {'city': 'SPb', 'occu

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [ ]:
from typing import Iterator, NamedTuple, Any, Tuple, List
import random

class User(NamedTuple):
    id: int
    name: str
    age: int

R = [
    User(id=1, name="Alice", age=25),
    User(id=2, name="Bob", age=30),
    User(id=3, name="Charlie", age=35),
]

S = [
    User(id=3, name="Charlie", age=35),
    User(id=4, name="Diana", age=28),
    User(id=5, name="Eve", age=32),
    User(id=2, name="Bob", age=30),
]

print("Отношение R:")
for user in R:
    print(f"  {user}")

print("\nОтношение S:")
for user in S:
    print(f"  {user}")

def RECORDREADER():
    for user in R:
        yield (f"R_{user.id}", user)

    for user in S:
        yield (f"S_{user.id}", user)

def union_MAP(key: str, user: User):
    yield (user, user)

def union_REDUCE(user: User, values: Iterator[User]):
    yield (user, user)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x),
                       groupbykey(flatten(map(lambda x: MAP(*x),
                                             RECORDREADER())))))

print("UNION R ∪ S")
result = list(MapReduce(RECORDREADER, union_MAP, union_REDUCE))
print(f"\nРезультат объединения ({len(result)} записей):")
for user_key, user_value in sorted(result, key=lambda x: x[0].id):
    print(f"  {user_key}")

Отношение R:
  User(id=1, name='Alice', age=25)
  User(id=2, name='Bob', age=30)
  User(id=3, name='Charlie', age=35)

Отношение S:
  User(id=3, name='Charlie', age=35)
  User(id=4, name='Diana', age=28)
  User(id=5, name='Eve', age=32)
  User(id=2, name='Bob', age=30)
UNION R ∪ S

Результат объединения (5 записей):
  User(id=1, name='Alice', age=25)
  User(id=2, name='Bob', age=30)
  User(id=3, name='Charlie', age=35)
  User(id=4, name='Diana', age=28)
  User(id=5, name='Eve', age=32)


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
from typing import Iterator, NamedTuple, Any, Tuple, List
from collections import Counter

class User(NamedTuple):
    id: int
    name: str
    age: int

R = [
    User(id=1, name="Alice", age=25),
    User(id=2, name="Bob", age=30),
    User(id=3, name="Charlie", age=35),
    User(id=4, name="Diana", age=28),
]

S = [
    User(id=3, name="Charlie", age=35),
    User(id=4, name="Diana", age=28),
    User(id=5, name="Eve", age=32),
    User(id=6, name="Frank", age=40),
    User(id=2, name="Bob", age=30),
]

print("Отношение R:")
for user in R:
    print(f"  {user}")

print("\nОтношение S:")
for user in S:
    print(f"  {user}")

print(f"\nОжидаемое пересечение R ∩ S:")
expected_intersection = set(R) & set(S)
for user in sorted(expected_intersection, key=lambda x: x.id):
    print(f"  {user}")

def RECORDREADER():
    for user in R:
        yield (('R', user.id), user)

    for user in S:
        yield (('S', user.id), user)

def intersection_MAP(key: tuple, user: User):
    source = key[0]  # 'R' или 'S'
    yield (user, (user, source))

def intersection_REDUCE(user: User, values: Iterator[Tuple[User, str]]):
    values_list = list(values)
    sources = [source for (_, source) in values_list]

    if 'R' in sources and 'S' in sources:
        yield (user, user)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x),
                       groupbykey(flatten(map(lambda x: MAP(*x),
                                             RECORDREADER())))))

print("INTERSECTION R ∩ S")

result = list(MapReduce(RECORDREADER, intersection_MAP, intersection_REDUCE))
print(f"\nРезультат пересечения ({len(result)} записей):")
for user_key, user_value in sorted(result, key=lambda x: x[0].id):
    print(f"  {user_key}")

Отношение R:
  User(id=1, name='Alice', age=25)
  User(id=2, name='Bob', age=30)
  User(id=3, name='Charlie', age=35)
  User(id=4, name='Diana', age=28)

Отношение S:
  User(id=3, name='Charlie', age=35)
  User(id=4, name='Diana', age=28)
  User(id=5, name='Eve', age=32)
  User(id=6, name='Frank', age=40)
  User(id=2, name='Bob', age=30)

Ожидаемое пересечение R ∩ S:
  User(id=2, name='Bob', age=30)
  User(id=3, name='Charlie', age=35)
  User(id=4, name='Diana', age=28)
INTERSECTION R ∩ S

Результат пересечения (3 записей):
  User(id=2, name='Bob', age=30)
  User(id=3, name='Charlie', age=35)
  User(id=4, name='Diana', age=28)


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [ ]:
from typing import Iterator, NamedTuple, Any, Tuple, List, Set
from collections import Counter

class User(NamedTuple):
    id: int
    name: str
    age: int

R = [
    User(id=1, name="Alice", age=25),
    User(id=2, name="Bob", age=30),
    User(id=3, name="Charlie", age=35),
    User(id=4, name="Diana", age=28),
    User(id=7, name="Grace", age=22),
]

S = [
    User(id=3, name="Charlie", age=35),  # есть в R
    User(id=4, name="Diana", age=28),    # есть в R
    User(id=5, name="Eve", age=32),      # нет в R
    User(id=6, name="Frank", age=40),    # нет в R
    User(id=2, name="Bob", age=30),      # есть в R
]

print("Отношение R:")
for user in R:
    print(f"  {user}")

print("\nОтношение S:")
for user in S:
    print(f"  {user}")

print(f"\nОжидаемая разность R - S (только в R, не в S):")
expected_difference = set(R) - set(S)
for user in sorted(expected_difference, key=lambda x: x.id):
    print(f"  {user}")

def RECORDREADER():
    for user in R:
        yield (('R', user.id), user)

    for user in S:
        yield (('S', user.id), user)

def difference_MAP(key: tuple, user: User):
    source = key[0]  # 'R' или 'S'
    yield (user, source)

def difference_REDUCE(user: User, sources: Iterator[str]):
    sources_set = set(sources)

    if 'R' in sources_set and 'S' not in sources_set:
        yield (user, user)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x),
                       groupbykey(flatten(map(lambda x: MAP(*x),
                                             RECORDREADER())))))

print("DIFFERENCE R - S (только в R, не в S)")

result = list(MapReduce(RECORDREADER, difference_MAP, difference_REDUCE))
print(f"\nРезультат разности ({len(result)} записей):")
for user_key, user_value in sorted(result, key=lambda x: x[0].id):
    print(f"  {user_key}")

Отношение R:
  User(id=1, name='Alice', age=25)
  User(id=2, name='Bob', age=30)
  User(id=3, name='Charlie', age=35)
  User(id=4, name='Diana', age=28)
  User(id=7, name='Grace', age=22)

Отношение S:
  User(id=3, name='Charlie', age=35)
  User(id=4, name='Diana', age=28)
  User(id=5, name='Eve', age=32)
  User(id=6, name='Frank', age=40)
  User(id=2, name='Bob', age=30)

Ожидаемая разность R - S (только в R, не в S):
  User(id=1, name='Alice', age=25)
  User(id=7, name='Grace', age=22)
DIFFERENCE R - S (только в R, не в S)

Результат разности (2 записей):
  User(id=1, name='Alice', age=25)
  User(id=7, name='Grace', age=22)


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [ ]:
from typing import Iterator, NamedTuple, Any, Tuple, List
from itertools import product

class R(NamedTuple):
    a: str  # первый атрибут
    b: int  # общий атрибут для соединения

class S(NamedTuple):
    b: int  # общий атрибут для соединения
    c: str  # третий атрибут

relation_R = [
    R(a="Alice", b=25),
    R(a="Bob", b=30),
    R(a="Charlie", b=35),
    R(a="Diana", b=25),
    R(a="Eve", b=30),
]

relation_S = [
    S(b=25, c="Engineer"),
    S(b=25, c="Manager"),   # два разных c для b=25
    S(b=30, c="Doctor"),
    S(b=35, c="Teacher"),
    S(b=40, c="Lawyer"),    # нет в R
]

print("Отношение R (a, b):")
for row in relation_R:
    print(f"  {row}")

print("\nОтношение S (b, c):")
for row in relation_S:
    print(f"  {row}")

print("\nОжидаемый результат Natural Join R ⨝ S:")
for r in relation_R:
    for s in relation_S:
        if r.b == s.b:
            print(f"  ({r.a}, {r.b}, {s.c})")

def RECORDREADER():
    for row in relation_R:
        yield (('R', row.b), row)

    for row in relation_S:
        yield (('S', row.b), row)

def natural_join_MAP(key: tuple, value: Any):
    source = key[0]  # 'R' или 'S'

    if source == 'R':
        # Для R: (b, ('R', a))
        yield (value.b, ('R', value.a))
    else:
        # Для S: (b, ('S', c))
        yield (value.b, ('S', value.c))

def natural_join_REDUCE(b: int, values: Iterator[Tuple[str, Any]]):
    r_values = []
    s_values = []

    for source, val in values:
        if source == 'R':
            r_values.append(val)  # val это a
        else:
            s_values.append(val)  # val это c

    for a in r_values:
        for c in s_values:
            yield ((a, b, c), (a, b, c))

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x),
                       groupbykey(flatten(map(lambda x: MAP(*x),
                                             RECORDREADER())))))


print("NATURAL JOIN R ⨝ S")

result = list(MapReduce(RECORDREADER, natural_join_MAP, natural_join_REDUCE))
print(f"\nРезультат соединения ({len(result)} записей):")
for (a, b, c), _ in sorted(result, key=lambda x: (x[0][1], x[0][0])):
    print(f"  ({a}, {b}, {c})")

Отношение R (a, b):
  R(a='Alice', b=25)
  R(a='Bob', b=30)
  R(a='Charlie', b=35)
  R(a='Diana', b=25)
  R(a='Eve', b=30)

Отношение S (b, c):
  S(b=25, c='Engineer')
  S(b=25, c='Manager')
  S(b=30, c='Doctor')
  S(b=35, c='Teacher')
  S(b=40, c='Lawyer')

Ожидаемый результат Natural Join R ⨝ S:
  (Alice, 25, Engineer)
  (Alice, 25, Manager)
  (Bob, 30, Doctor)
  (Charlie, 35, Teacher)
  (Diana, 25, Engineer)
  (Diana, 25, Manager)
  (Eve, 30, Doctor)
NATURAL JOIN R ⨝ S

Результат соединения (7 записей):
  (Alice, 25, Engineer)
  (Alice, 25, Manager)
  (Diana, 25, Engineer)
  (Diana, 25, Manager)
  (Bob, 30, Doctor)
  (Eve, 30, Doctor)
  (Charlie, 35, Teacher)


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [ ]:
from typing import Iterator, NamedTuple, Any, Tuple, List, Callable
from statistics import mean, median, stdev
import math

class Sale(NamedTuple):
    product: str    # a - группа (продукт)
    amount: int     # b - значение для агрегации
    store: str

sales_data = [
    Sale(product="Apple", amount=100, store="Store1"),
    Sale(product="Apple", amount=150, store="Store2"),
    Sale(product="Apple", amount=200, store="Store3"),
    Sale(product="Banana", amount=80, store="Store1"),
    Sale(product="Banana", amount=120, store="Store2"),
    Sale(product="Banana", amount=90, store="Store3"),
    Sale(product="Orange", amount=300, store="Store1"),
    Sale(product="Orange", amount=250, store="Store2"),
    Sale(product="Apple", amount=175, store="Store4"),
    Sale(product="Banana", amount=110, store="Store4"),
    Sale(product="Grape", amount=50, store="Store1"),
    Sale(product="Grape", amount=60, store="Store2"),
]

print("Исходные данные (product, amount, store):")
for sale in sales_data:
    print(f"  {sale}")

def RECORDREADER():
    for i, sale in enumerate(sales_data):
        yield (i, sale)

def grouping_MAP(key: int, sale: Sale):
    yield (sale.product, sale.amount)

def sum_aggregator(values: Iterator[int]) -> int:
    return sum(values)

def count_aggregator(values: Iterator[int]) -> int:
    return len(list(values))

def avg_aggregator(values: Iterator[int]) -> float:
    vals = list(values)
    return sum(vals) / len(vals) if vals else 0

def max_aggregator(values: Iterator[int]) -> int:
    return max(values)

def min_aggregator(values: Iterator[int]) -> int:
    return min(values)

def create_aggregation_REDUCE(aggregator_func: Callable):
    def REDUCE(product: str, amounts: Iterator[int]):
        result = aggregator_func(amounts)
        yield (product, result)
    return REDUCE

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x),
                       groupbykey(flatten(map(lambda x: MAP(*x),
                                             RECORDREADER())))))

print("SUM - сумма продаж по продуктам")

sum_REDUCE = create_aggregation_REDUCE(sum_aggregator)
result_sum = list(MapReduce(RECORDREADER, grouping_MAP, sum_REDUCE))

print("Результат:")
for product, total in sorted(result_sum):
    print(f"  {product}: {total}")

print("\n")
print("COUNT - количество продаж по продуктам")

count_REDUCE = create_aggregation_REDUCE(count_aggregator)
result_count = list(MapReduce(RECORDREADER, grouping_MAP, count_REDUCE))

print("Результат:")
for product, count in sorted(result_count):
    print(f"  {product}: {count} продаж")

print("\n")
print("AVG - средняя сумма продажи по продуктам")

avg_REDUCE = create_aggregation_REDUCE(avg_aggregator)
result_avg = list(MapReduce(RECORDREADER, grouping_MAP, avg_REDUCE))

print("Результат:")
for product, average in sorted(result_avg):
    print(f"  {product}: {average:.2f}")

print("\n")
print("MAX и MIN - экстремумы по продуктам")

max_REDUCE = create_aggregation_REDUCE(max_aggregator)
min_REDUCE = create_aggregation_REDUCE(min_aggregator)

result_max = list(MapReduce(RECORDREADER, grouping_MAP, max_REDUCE))
result_min = list(MapReduce(RECORDREADER, grouping_MAP, min_REDUCE))

print("MAX значения:")
for product, max_val in sorted(result_max):
    print(f"  {product}: {max_val}")

print("\nMIN значения:")
for product, min_val in sorted(result_min):
    print(f"  {product}: {min_val}")

Исходные данные (product, amount, store):
  Sale(product='Apple', amount=100, store='Store1')
  Sale(product='Apple', amount=150, store='Store2')
  Sale(product='Apple', amount=200, store='Store3')
  Sale(product='Banana', amount=80, store='Store1')
  Sale(product='Banana', amount=120, store='Store2')
  Sale(product='Banana', amount=90, store='Store3')
  Sale(product='Orange', amount=300, store='Store1')
  Sale(product='Orange', amount=250, store='Store2')
  Sale(product='Apple', amount=175, store='Store4')
  Sale(product='Banana', amount=110, store='Store4')
  Sale(product='Grape', amount=50, store='Store1')
  Sale(product='Grape', amount=60, store='Store2')
SUM - сумма продаж по продуктам
Результат:
  Apple: 625
  Banana: 400
  Grape: 110
  Orange: 550


COUNT - количество продаж по продуктам
Результат:
  Apple: 4 продаж
  Banana: 4 продаж
  Grape: 2 продаж
  Orange: 2 продаж


AVG - средняя сумма продажи по продуктам
Результат:
  Apple: 156.25
  Banana: 100.00
  Grape: 55.00
  Orang

#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [ ]:
from typing import Iterator, Tuple, List, Dict
import numpy as np
import random

matrix_size = (100, 1000)
vector_size = matrix_size[1]

matrix = np.random.rand(matrix_size[0], matrix_size[1])
vector = np.random.rand(vector_size)

print(f"Размер матрицы: {matrix_size[0]}x{matrix_size[1]}")
print(f"Размер вектора: {vector_size}")
print(f"Вектор слишком большой, чтобы поместиться в памяти каждого маппера!")

# Параметры распределённой системы
maps = 4  # количество мапперов
reducers = 2  # количество редьюсеров

class MatrixChunk:
    def __init__(self, chunk_id: int, rows: List[int], data: Dict):
        self.chunk_id = chunk_id
        self.rows = rows
        self.data = data

def create_distributed_vector():
    vector_chunks = {}
    chunk_size = vector_size // maps + 1

    for i in range(0, vector_size, chunk_size):
        chunk_id = i // chunk_size
        chunk_data = {}
        for j in range(i, min(i + chunk_size, vector_size)):
            chunk_data[j] = vector[j]
        vector_chunks[chunk_id] = chunk_data

    return vector_chunks

vector_chunks = create_distributed_vector()
print(f"\nВектор разбит на {len(vector_chunks)} частей:")
for chunk_id, chunk in vector_chunks.items():
    print(f"  Часть {chunk_id}: {len(chunk)} элементов (индексы {min(chunk.keys())}-{max(chunk.keys())})")

def INPUTFORMAT():
    global maps

    rows_per_map = matrix_size[0] // maps + 1

    for map_id in range(maps):
        start_row = map_id * rows_per_map
        end_row = min(start_row + rows_per_map, matrix_size[0])

        def RECORDREADER():
            for i in range(start_row, end_row):
                for j in range(matrix_size[1]):
                    if matrix[i, j] != 0:
                        yield ((i, j), matrix[i, j])

        print(f"  Маппер {map_id} будет обрабатывать строки {start_row}-{end_row-1}")
        yield RECORDREADER

def MAP(key: Tuple[int, int], value: float):
    row, col = key

    vector_value = get_vector_element(col)

    partial_product = value * vector_value

    yield (row, partial_product)

def get_vector_element(index: int) -> float:
    for chunk_id, chunk in vector_chunks.items():
        if index in chunk:
            return chunk[index]
    return 0.0

def PARTITIONER(row: int):
    global reducers
    return row % reducers

def COMBINER(row: int, partial_products: Iterator[float]):
    total = sum(partial_products)
    yield (row, total)

def REDUCE(row: int, partial_sums: Iterator[float]):
    total = sum(partial_sums)
    yield (row, total)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
    global reducers
    partitions = [dict() for _ in range(reducers)]
    for map_partition in map_partitions:
        for (k2, v2) in map_partition:
            p = partitions[PARTITIONER(k2)]
            p[k2] = p.get(k2, []) + [v2]
    return [(partition_id, sorted(partition.items(), key=lambda x: x[0]))
            for (partition_id, partition) in enumerate(partitions)]

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=None):
    map_partitions = []
    for record_reader in INPUTFORMAT():
        map_output = list(flatten(map(lambda x: MAP(*x), record_reader())))
        map_partitions.append(map_output)

        print(f"  Маппер обработал {len(map_output)} ненулевых элементов")

    if COMBINER:
        combined_partitions = []
        for map_partition in map_partitions:
            combined = list(flatten(map(lambda x: COMBINER(*x), groupbykey(map_partition))))
            combined_partitions.append(combined)
            print(f"  Комбайнер сжал данные до {len(combined)} пар")
        map_partitions = combined_partitions

    reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER)

    network_pairs = sum([len(vs) for (k,vs) in
                        flatten([partition for (partition_id, partition) in reduce_partitions])])
    print(f"\nПо сети передано: {network_pairs} пар")

    reduce_outputs = map(lambda reduce_partition:
                        (reduce_partition[0],
                         list(flatten(map(lambda reduce_input_group:
                                    REDUCE(*reduce_input_group),
                                    reduce_partition[1])))),
                        reduce_partitions)

    return reduce_outputs

print("ЗАПУСК УМНОЖЕНИЯ МАТРИЦЫ НА ВЕКТОР (ВЕКТОР НЕ В ПАМЯТИ)")

print(f"\nКонфигурация:")
print(f"  Мапперов: {maps}")
print(f"  Редьюсеров: {reducers}")
print(f"  Размер матрицы: {matrix_size[0]}x{matrix_size[1]}")
print(f"  Размер вектора: {vector_size}")

print("\nВыполнение MapReduce:")
result = MapReduceDistributed(
    INPUTFORMAT,
    MAP,
    REDUCE,
    PARTITIONER,
    COMBINER=COMBINER
)

print("\n")
print("РЕЗУЛЬТАТЫ ПО РЕДЬЮСЕРАМ")

final_result = {}
for partition_id, partition_output in result:
    print(f"\nРедьюсер {partition_id}:")
    for row, value in partition_output:
        print(f"  строка {row}: {value:.6f}")
        final_result[row] = value

print("\n")
print("ПРОВЕРКА РЕЗУЛЬТАТА")

expected = np.dot(matrix, vector)
print(f"Размер результата: {len(expected)}")
print(f"Первые 5 элементов ожидаемого результата: {expected[:5]}")

print(f"\nПервые 5 элементов полученного результата:")
for i in range(min(5, len(final_result))):
    print(f"  строка {i}: {final_result[i]:.6f}")

print(f"\nМаксимальная ошибка: {np.max(np.abs(expected - [final_result[i] for i in range(len(expected))])):.2e}")

Размер матрицы: 100x1000
Размер вектора: 1000
Вектор слишком большой, чтобы поместиться в памяти каждого маппера!

Вектор разбит на 4 частей:
  Часть 0: 251 элементов (индексы 0-250)
  Часть 1: 251 элементов (индексы 251-501)
  Часть 2: 251 элементов (индексы 502-752)
  Часть 3: 247 элементов (индексы 753-999)
ЗАПУСК УМНОЖЕНИЯ МАТРИЦЫ НА ВЕКТОР (ВЕКТОР НЕ В ПАМЯТИ)

Конфигурация:
  Мапперов: 4
  Редьюсеров: 2
  Размер матрицы: 100x1000
  Размер вектора: 1000

Выполнение MapReduce:
  Маппер 0 будет обрабатывать строки 0-25
  Маппер обработал 26000 ненулевых элементов
  Маппер 1 будет обрабатывать строки 26-51
  Маппер обработал 26000 ненулевых элементов
  Маппер 2 будет обрабатывать строки 52-77
  Маппер обработал 26000 ненулевых элементов
  Маппер 3 будет обрабатывать строки 78-99
  Маппер обработал 22000 ненулевых элементов
  Комбайнер сжал данные до 26 пар
  Комбайнер сжал данные до 26 пар
  Комбайнер сжал данные до 26 пар
  Комбайнер сжал данные до 22 пар

По сети передано: 100 пар


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [ ]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [ ]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  # solution code that yield(k2,v2) pairs

def REDUCE(key, values):
  (i, k) = key
  # solution code that yield(k3,v3) pairs

Проверьте своё решение

In [ ]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [ ]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [1]:
from typing import Iterator, Tuple
import numpy as np

I, J, K = 3, 4, 5  # M: I×J, N: J×K

M = np.random.randint(1, 5, size=(I, J))
N = np.random.randint(1, 5, size=(J, K))

print("Матрица M:")
print(M)
print("\nМатрица N:")
print(N)
print("\nОжидаемый результат M × N:")
print(np.dot(M, N))

def RECORDREADER():
    for i in range(I):
        for j in range(J):
            yield (('M', i, j), M[i, j])

    for j in range(J):
        for k in range(K):
            yield (('N', j, k), N[j, k])

def MAP(key: tuple, value: int):
    matrix_type = key[0]

    if matrix_type == 'M':
        i, j = key[1], key[2]
        yield (j, ('M', i, value))
    else:  # 'N'
        j, k = key[1], key[2]
        yield (j, ('N', k, value))

def REDUCE(j: int, values: Iterator[tuple]):
    m_entries = []  # (i, value)
    n_entries = []  # (k, value)

    for val in values:
        if val[0] == 'M':
            m_entries.append((val[1], val[2]))
        else:  # 'N'
            n_entries.append((val[1], val[2]))

    for i, m_val in m_entries:
        for k, n_val in n_entries:
            yield ((i, k), m_val * n_val)

def flatten(nested_iterable):
    for iterable in nested_iterable:
        for element in iterable:
            yield element

def groupbykey(iterable):
    t = {}
    for (k2, v2) in iterable:
        t[k2] = t.get(k2, []) + [v2]
    return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
    map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))

    grouped = groupbykey(map_output)

    result = flatten(map(lambda x: REDUCE(*x), grouped))

    return list(result)

result = MapReduce(RECORDREADER, MAP, REDUCE)

P = np.zeros((I, K))
for (i, k), value in result:
    P[i, k] += value

print("\n" + "="*50)
print("РЕЗУЛЬТАТ УМНОЖЕНИЯ:")
print(P)
print(f"\nРезультаты совпадают: {np.allclose(P, np.dot(M, N))}")

Матрица M:
[[1 4 3 2]
 [2 1 4 1]
 [1 3 3 2]]

Матрица N:
[[1 4 3 3 3]
 [2 3 4 2 1]
 [4 1 2 2 2]
 [1 3 2 4 1]]

Ожидаемый результат M × N:
[[23 25 29 25 15]
 [21 18 20 20 16]
 [21 22 25 23 14]]

РЕЗУЛЬТАТ УМНОЖЕНИЯ:
[[23. 25. 29. 25. 15.]
 [21. 18. 20. 20. 16.]
 [21. 22. 25. 23. 14.]]

Результаты совпадают: True


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [11]:
from typing import Iterator, Tuple
import numpy as np

I, J, K = 3, 4, 5  # M: 3×4, N: 4×5

np.random.seed(42)
M = np.random.randint(1, 10, size=(I, J))
N = np.random.randint(1, 10, size=(J, K))

print("Матрица M:")
print(M)
print("\nМатрица N:")
print(N)

print("\nОжидаемый результат M × N:")
expected = np.dot(M, N)
print(expected)

# Параметры распределённой системы
reducers = 2

def INPUTFORMAT():
    def M_RECORDREADER():
        for i in range(I):
            for j in range(J):
                yield (('M', i, j), M[i, j])

    def N_RECORDREADER():
        for j in range(J):
            for k in range(K):
                yield (('N', j, k), N[j, k])

    return [M_RECORDREADER, N_RECORDREADER]

def MAP(key: tuple, value: int):
    matrix_type = key[0]

    if matrix_type == 'M':
        i, j = key[1], key[2]
        yield (j, ('M', i, value))
    else:  # 'N'
        j, k = key[1], key[2]
        yield (j, ('N', k, value))

def PARTITIONER(key: int):
    return key % reducers

def COMBINER(j: int, values: list):
    return [(j, v) for v in values]

def REDUCE(j: int, values: list):
    m_entries = []  # (i, value)
    n_entries = []  # (k, value)

    for val in values:
        if val[0] == 'M':
            m_entries.append((val[1], val[2]))
        else:  # 'N'
            n_entries.append((val[1], val[2]))

    result = []
    for i, m_val in m_entries:
        for k, n_val in n_entries:
            result.append(((i, k), m_val * n_val))

    return result

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER=None):
    record_readers = INPUTFORMAT()

    map_outputs = []
    for reader in record_readers:
        map_output = []
        for key, value in reader():
            for mapped in MAP(key, value):
                map_output.append(mapped)
        map_outputs.append(map_output)

    if COMBINER:
        combined_outputs = []
        for map_output in map_outputs:
            groups = {}
            for k, v in map_output:
                if k not in groups:
                    groups[k] = []
                groups[k].append(v)

            combined = []
            for k, values in groups.items():
                combined.extend(COMBINER(k, values))
            combined_outputs.append(combined)
        map_outputs = combined_outputs

    reducer_inputs = [[] for _ in range(reducers)]
    for map_output in map_outputs:
        for j, val in map_output:
            reducer_id = PARTITIONER(j)
            reducer_inputs[reducer_id].append((j, val))

    final_result = {}
    for reducer_id, inputs in enumerate(reducer_inputs):
        groups = {}
        for j, val in inputs:
            if j not in groups:
                groups[j] = []
            groups[j].append(val)

        for j, values in groups.items():
            for (i, k), product in REDUCE(j, values):
                final_result[(i, k)] = final_result.get((i, k), 0) + product

    return final_result

final_result = MapReduceDistributed(
    INPUTFORMAT,
    MAP,
    REDUCE,
    PARTITIONER,
    COMBINER=COMBINER
)

P_result = np.zeros((I, K))
for (i, k), value in final_result.items():
    P_result[i, k] = value

print("\n" + "="*50)
print("РЕЗУЛЬТАТ УМНОЖЕНИЯ (С COMBINER):")
print(P_result)

print(f"\nРезультаты совпадают: {np.allclose(P_result, expected)}")

Матрица M:
[[7 4 8 5]
 [7 3 7 8]
 [5 4 8 8]]

Матрица N:
[[3 6 5 2 8]
 [6 2 5 1 6]
 [9 1 3 7 4]
 [9 3 5 3 7]]

Ожидаемый результат M × N:
[[162  73 104  89 147]
 [174  79 111  90 158]
 [183  70 109  94 152]]

РЕЗУЛЬТАТ УМНОЖЕНИЯ (С COMBINER):
[[162.  73. 104.  89. 147.]
 [174.  79. 111.  90. 158.]
 [183.  70. 109.  94. 152.]]

Результаты совпадают: True


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [17]:
from typing import Iterator, Tuple, List
import numpy as np

I, J, K = 4, 5, 6  # M: 4×5, N: 5×6

np.random.seed(42)
M = np.random.randint(1, 5, size=(I, J))
N = np.random.randint(1, 5, size=(J, K))

print("Матрица M:")
print(M)
print("\nМатрица N:")
print(N)

print("\nОжидаемый результат M × N:")
expected = np.dot(M, N)
print(expected)

reducers = 3
maps_per_matrix = 2

def INPUTFORMAT():
    readers = []

    rows_per_reader = I // maps_per_matrix + 1
    for reader_id in range(maps_per_matrix):
        start_row = reader_id * rows_per_reader
        end_row = min(start_row + rows_per_reader, I)

        def M_RECORDREADER(start=start_row, end=end_row):
            for i in range(start, end):
                for j in range(J):
                    if np.random.random() < 0.8:
                        yield (('M', i, j), M[i, j])

        readers.append(M_RECORDREADER)

    cols_per_reader = K // maps_per_matrix + 1
    for reader_id in range(maps_per_matrix):
        start_col = reader_id * cols_per_reader
        end_col = min(start_col + cols_per_reader, K)

        def N_RECORDREADER(start=start_col, end=end_col):
            for j in range(J):
                for k in range(start, end):
                    if np.random.random() < 0.8:
                        yield (('N', j, k), N[j, k])

        readers.append(N_RECORDREADER)
    return readers

def MAP(key: tuple, value: int):
    matrix_type = key[0]

    if matrix_type == 'M':
        i, j = key[1], key[2]
        yield (j, ('M', i, value))
    else:  # 'N'
        j, k = key[1], key[2]
        yield (j, ('N', k, value))

def PARTITIONER(key: int):
    return key % reducers

def REDUCE(j: int, values: list):
    m_dict = {}
    n_dict = {}

    for val in values:
        if val[0] == 'M':
            i, m_val = val[1], val[2]
            m_dict[i] = m_dict.get(i, 0) + m_val
        else:  # 'N'
            k, n_val = val[1], val[2]
            n_dict[k] = n_dict.get(k, 0) + n_val

    result = []
    for i, m_sum in m_dict.items():
        for k, n_sum in n_dict.items():
            result.append(((i, k), m_sum * n_sum))

    return result

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER):
    record_readers = INPUTFORMAT()

    map_outputs = []
    for reader_idx, reader in enumerate(record_readers):
        map_output = []
        for key, value in reader():
            for mapped in MAP(key, value):
                map_output.append(mapped)
        map_outputs.append(map_output)

    reducer_inputs = [[] for _ in range(reducers)]
    for map_output in map_outputs:
        for j, val in map_output:
            reducer_id = PARTITIONER(j)
            reducer_inputs[reducer_id].append((j, val))

    final_result = {}
    for reducer_id, inputs in enumerate(reducer_inputs):
        groups = {}
        for j, val in inputs:
            if j not in groups:
                groups[j] = []
            groups[j].append(val)

        for j, values in groups.items():
            for (i, k), product in REDUCE(j, values):
                final_result[(i, k)] = final_result.get((i, k), 0) + product

    return final_result

print("ПОЛНЫЕ МАТРИЦЫ")

def INPUTFORMAT_FULL():
    readers = []

    rows_per_reader = I // maps_per_matrix + 1
    for reader_id in range(maps_per_matrix):
        start_row = reader_id * rows_per_reader
        end_row = min(start_row + rows_per_reader, I)

        def M_RECORDREADER(start=start_row, end=end_row):
            for i in range(start, end):
                for j in range(J):
                    yield (('M', i, j), M[i, j])
        readers.append(M_RECORDREADER)

    cols_per_reader = K // maps_per_matrix + 1
    for reader_id in range(maps_per_matrix):
        start_col = reader_id * cols_per_reader
        end_col = min(start_col + cols_per_reader, K)

        def N_RECORDREADER(start=start_col, end=end_col):
            for j in range(J):
                for k in range(start, end):
                    yield (('N', j, k), N[j, k])
        readers.append(N_RECORDREADER)

    return readers

final_result_full = MapReduceDistributed(INPUTFORMAT_FULL, MAP, REDUCE, PARTITIONER)

P_full = np.zeros((I, K))
for (i, k), value in final_result_full.items():
    P_full[i, k] = value

print("\nРЕЗУЛЬТАТ (полные матрицы):")
print(P_full)
print(f"\nСовпадает с ожидаемым: {np.allclose(P_full, expected)}")

print("\n" + "="*70)
print("СЛУЧАЙНЫЕ ПОДМНОЖЕСТВА")

final_result_random = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER)

P_random = np.zeros((I, K))
for (i, k), value in final_result_random.items():
    P_random[i, k] = value

print("\nРЕЗУЛЬТАТ (случайные подмножества):")
print(P_random)
print(f"\nСовпадает с ожидаемым: {np.allclose(P_random, expected)}")

Матрица M:
[[3 4 1 3 3]
 [4 1 1 3 2]
 [3 3 3 3 4]
 [1 4 4 4 3]]

Матрица N:
[[2 1 2 4 4 2]
 [2 2 4 4 1 1]
 [4 2 2 1 4 1]
 [1 3 3 3 2 4]
 [4 4 4 3 2 2]]

Ожидаемый результат M × N:
[[33 34 45 47 32 29]
 [25 25 31 36 31 26]
 [43 40 49 48 41 32]
 [42 41 50 45 38 32]]
ПОЛНЫЕ МАТРИЦЫ

РЕЗУЛЬТАТ (полные матрицы):
[[33. 34. 45. 47. 32. 29.]
 [25. 25. 31. 36. 31. 26.]
 [43. 40. 49. 48. 41. 32.]
 [42. 41. 50. 45. 38. 32.]]

Совпадает с ожидаемым: True

СЛУЧАЙНЫЕ ПОДМНОЖЕСТВА

РЕЗУЛЬТАТ (случайные подмножества):
[[29. 14. 43. 31. 18. 29.]
 [10.  6. 12. 17. 16. 10.]
 [25. 15. 37. 24.  6. 26.]
 [22.  9. 30. 17.  4. 16.]]

Совпадает с ожидаемым: False
